# Supervised Tabular Training

This notebook trains a compact flat classifier on the bundled Wine dataset. It follows the same API as Hello World, but uses a few more numeric inputs and exposes root embeddings during the same run. Use it as a tabular comparison point, not as the main nested-data story.

Import the runtime pieces used in the full training loop: Lightning for optimization, Polars for tabular records, and scikit-learn for a local dataset.

In [1]:
import contextlib
import io
import logging

import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger
from rich.pretty import pprint
from sklearn.datasets import load_wine

import json2vec as j2v

logger.remove()

The Wine dataset is still flat, but it gives enough numeric variation to make a clearer supervised example than handmade records. The model will use four chemical measurements to predict `cultivar`.

In [2]:
wine = load_wine()
indices = []
for target_id in range(len(wine.target_names)):
    indices.extend(index for index, target in enumerate(wine.target) if target == target_id)
indices = indices[:48]

records = pl.DataFrame(
    {
        "alcohol": wine.data[indices, 0].tolist(),
        "malic_acid": wine.data[indices, 1].tolist(),
        "color_intensity": wine.data[indices, 9].tolist(),
        "proline": wine.data[indices, 12].tolist(),
        "cultivar": [wine.target_names[index] for index in wine.target[indices]],
    }
)

records.head()

alcohol,malic_acid,color_intensity,proline,cultivar
f64,f64,f64,f64,str
14.23,1.71,5.64,1065.0,"""class_0"""
13.2,1.78,4.38,1050.0,"""class_0"""
13.16,2.36,5.68,1185.0,"""class_0"""
14.37,1.95,7.8,1480.0,"""class_0"""
13.24,2.59,4.32,735.0,"""class_0"""


The schema is the architecture. Four `Number` requests feed the root encoder, `cultivar` is the categorical target, and `model.set(..., embed=True)` asks the root node to return embeddings after training.

In [3]:
model = j2v.Model.from_schema(
    j2v.Number("alcohol"),
    j2v.Number("malic_acid"),
    j2v.Number("color_intensity"),
    j2v.Number("proline"),
    j2v.Category("cultivar", target=True, max_vocab_size=4, topk=[2]),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)
_ = model.set(j2v.where("name") == "record", embed=True)

`PolarsDataModule.from_model(...)` reads the schema configuration from the model, so batch size, queries, targets, and tensorfield behavior stay tied to one object.

In [4]:
datamodule = j2v.PolarsDataModule.from_model(
    model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    chunk_batch_size=32,
    sample_rate=1.0,
)

The tutorial trains for one tiny pass. In a real experiment this is where you would scale epochs, validation splits, callbacks, logging, and checkpointing.

In [5]:
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    logging.disable(logging.CRITICAL)
    try:
        lit.seed_everything(7, workers=True, verbose=False)
        trainer = lit.Trainer(
            accelerator="cpu",
            max_epochs=1,
            logger=False,
            enable_progress_bar=False,
            enable_model_summary=False,
            enable_checkpointing=False,
            limit_train_batches=1,
            limit_val_batches=1,
        )
        trainer.fit(model=model, datamodule=datamodule)
    finally:
        logging.disable(logging.NOTSET)

The plot confirms that the target and root embedding are both configured before inference.

In [6]:
model.plot(detail=True)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                      │
│  JSON2Vec                                                             d_model  16                                    │
│  Model                                                                 params  26,039                                │
│  Schema map                                                            arrays  1                                     │
│                                                                        fields  5                                     │
│                                                                       targets  1                                     │
│                                                                        embeds  1                                     │
│                               

After training, `predict` returns typed outputs for supervised targets and `embed` returns vectors from the nodes configured with `embed=True`.

In [7]:
batch = [[record] for record in records.to_dicts()[:3]]
pprint(model.predict(batch))
pprint(model.embed(batch))

{
│   'record/cultivar': {
│   │   'state': {
│   │   │   'valued': [0.515616774559021, 0.5130342841148376, 0.5142344832420349],
│   │   │   'null': [0.15031996369361877, 0.1509433388710022, 0.15106801688671112],
│   │   │   'padded': [0.05998259037733078, 0.0591031014919281, 0.05963228642940521],
│   │   │   'masked': [0.12092842161655426, 0.1168593093752861, 0.11830899864435196],
│   │   │   'other': [0.1531522572040558, 0.16006000339984894, 0.15675613284111023]
│   │   },
│   │   'content': {
│   │   │   'value': ['class_0', 'class_0', 'class_0'],
│   │   │   'probability': [1.0, 1.0, 1.0],
│   │   │   'topk': [
│   │   │   │   [{'label': 'class_0', 'probability': 1.0}],
│   │   │   │   [{'label': 'class_0', 'probability': 1.0}],
│   │   │   │   [{'label': 'class_0', 'probability': 1.0}]
│   │   │   ]
│   │   }
│   }
}

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   -0.23039735853672028,
│   │   │   │   0.26419854164123535,
│   │   │   │   0.08428604900836945,
│   │   │   │   -0.2955012023448944,
│   │   │   │   -0.19986294209957123,
│   │   │   │   0.07435457408428192,
│   │   │   │   -0.25907132029533386,
│   │   │   │   -0.2760412096977234,
│   │   │   │   0.01341793593019247,
│   │   │   │   0.3307184875011444,
│   │   │   │   0.45432761311531067,
│   │   │   │   -0.0009946658974513412,
│   │   │   │   0.2207307666540146,
│   │   │   │   0.07885262370109558,
│   │   │   │   -0.4415126442909241,
│   │   │   │   0.16750532388687134
│   │   │   ],
│   │   │   [
│   │   │   │   -0.24860668182373047,
│   │   │   │   0.2678006887435913,
│   │   │   │   0.09802848100662231,
│   │   │   │   -0.2953192889690399,
│   │   │   │   -0.21291688084602356,
│   │   │   │   0.0934574156999588,
│   │   │   │   -0.25032132863998413,
│   │   │   │   -0.23424206674098969,
│   │   │   │   -0.007104739546775818,
│   │   │   │   0.31773391366004944,
│   │   │   │   0.44434455037117004,
│   │   │   │   -0.007099872455000877,
│   │   │   │   0.2236395627260208,
│   │   │   │   0.07087186723947525,
│   │   │   │   -0.45919010043144226,
│   │   │   │   0.18348222970962524
│   │   │   ],
│   │   │   [
│   │   │   │   -0.2447921484708786,
│   │   │   │   0.2699858546257019,
│   │   │   │   0.08436709642410278,
│   │   │   │   -0.2798483371734619,
│   │   │   │   -0.2264123409986496,
│   │   │   │   0.10293415188789368,
│   │   │   │   -0.24392803013324738,
│   │   │   │   -0.26105600595474243,
│   │   │   │   0.0036182361654937267,
│   │   │   │   0.3225003182888031,
│   │   │   │   0.4340568780899048,
│   │   │   │   -0.012900215573608875,
│   │   │   │   0.21768997609615326,
│   │   │   │   0.0892358347773552,
│   │   │   │   -0.4576946198940277,
│   │   │   │   0.18658751249313354
│   │   │   ]
│   │   ]
│   }
}